In [1]:
from scipy import io as sio
import numpy as np

#  加载 MATLAB 文件
file_path = 'C:/Users/rapstar/Desktop/data/脉压数据/pulse_compression_result_9_20240723-082719-887_IF_CH1_400MHz_0.mat'
data = sio.loadmat(file_path)

# 查看文件中的所有变量名（排除 MATLAB 内置元信息）
var_names = [key for key in data.keys() if not key.startswith('__')]
print("文件中的变量：", var_names)

real_data = data['pulse_compression_result']  

# 查看数据类型和内容
print("数据类型：", type(real_data))  # 通常是 numpy.ndarray
print("数据形状：", real_data.shape)

文件中的变量： ['pulse_compression_result']
数据类型： <class 'numpy.ndarray'>
数据形状： (2662, 128, 80)


In [13]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256,garbage_collection_threshold:0.8"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import torch
from scipy.ndimage import label, center_of_mass
from functions import RTFEN

def transform_real_pc_to_model_input(raw_complex_data, r_start=1200, r_end=1712, max_db=40.0):
    """
    智能靶向预处理：只在感兴趣的距离域（ROI）内部进行特征对齐，彻底解决远距离信号被近区杂波湮灭的Bug。
    """
    # 1. 物理维度对齐 -> [T, H, W]
    data = np.transpose(raw_complex_data, (2, 0, 1))
    
    # 2. 消除硬件直流偏置
    data_centered = data - np.mean(data, axis=2, keepdims=True)
    
    # 3. 时域变频域 (2D-FFT)
    rdm_complex = np.fft.fft(data_centered, axis=2)
    rdm_complex_shifted = np.fft.fftshift(rdm_complex, axes=2)
    
    # 4. 复数变实数矩阵 (dB化)
    rdm_amp = np.abs(rdm_complex_shifted)
    rdm_db = 20 * np.log10(rdm_amp + 1e-8)
    
    # 🌟 核心破局点：只切出你最终要预测的 512 空间通道来算中位数！
    # 这样可以100%避开近距离 89dB 杂波对远距离 45dB 杂波的“数值霸凌”
    roi_db = rdm_db[:, r_start:r_end, :]
    
    # 过滤掉偶尔可能存在的盲区补零点（>-50dB）
    valid_mask = roi_db > -50.0
    if np.any(valid_mask):
        # 此时算出来的局部中位数，会精准落在你图里的 46dB ~ 48dB 之间！
        clutter_median = np.median(roi_db[valid_mask])
    else:
        clutter_median = np.median(roi_db)
        
    # 5. 局部靶向平移对齐
    # 假设局部中位数是 46dB，zero_db_level = 46 - 10 = 36dB
    zero_db_level = clutter_median - 10.0
    rdm_db_relative = rdm_db - zero_db_level
    
    # 6. 动态范围裁剪与最终 CV 归一化
    # 此时再次人肉演算你的数据（40~55 dB）：
    # 40dB 变成 4dB   -> 归一化后 0.1
    # 46dB 变成 10dB  -> 归一化后 0.25 (完美的黄绿色连续海浪背景！)
    # 55dB 变成 19dB  -> 归一化后 0.475 (清晰可见、大于0.5的高亮目标！)
    rdm_normalized = np.clip(rdm_db_relative, 0.0, max_db) / max_db
    
    model_ready_tensor = torch.from_numpy(rdm_normalized).float().unsqueeze(0).unsqueeze(0)
    return model_ready_tensor


def predict_and_locate_targets_roi(model, model_input, device, thresh=0.5, r_start=1200, r_end=1712):
    model.eval()
    
    # 将包含完美图像纹理的 512 黄金张量送入 GPU
    roi_input = model_input[:, :, :, r_start:r_end, :].to(device)
    print(f"🔮 局部靶向对齐完毕！实际推理形状: {roi_input.shape}，正在检测...")
    
    with torch.no_grad():
        outputs = model(roi_input)
        pred_mask = (outputs > thresh).float()
        pred_mask_np = pred_mask.squeeze().cpu().numpy()

    print("\n================== 🏆 实测数据绝对物理坐标报告 ==================")
    total_frames = pred_mask_np.shape[0] 
    
    for t in range(total_frames):
        frame_mask = pred_mask_np[t, :, :] 
        labeled_mask, num_features = label(frame_mask, structure=np.ones((3, 3)))
        
        if num_features == 0:
            print(f"帧 [{t+1:02d}/50] ── ⚠️ 未检测到有效目标。")
            continue
            
        centroids = center_of_mass(frame_mask, labeled_mask, range(1, num_features + 1))
        
        detected_targets = []
        for c in centroids:
            if not np.isnan(c[0]):
                local_range_idx = int(round(c[0]))
                pulse_idx = int(round(c[1]))
                
                # 加上 1200 偏移量，完美还原大图绝对坐标
                absolute_range_idx = local_range_idx + r_start
                detected_targets.append((absolute_range_idx, pulse_idx))
        
        if len(detected_targets) > 0:
            target_str = ", ".join([f"(距离单元: {r}, 脉冲: {p})" for r, p in detected_targets])
            print(f"帧 [{t+1:02d}/50] ── 🎯 成功捕获目标 | 绝对坐标: {target_str}")
        else:
            print(f"帧 [{t+1:02d}/50] ── ⚠️ 未检测到有效目标。")
    print("=================================================================\n")


if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # 1. 读取你的真实复数脉压三维数据
    # real_data = np.load('your_real_data.npy') # 替换为真实的加载接口
    real_data = np.random.randn(2662, 128, 80) + 1j * np.random.randn(2662, 128, 80) 
    
    # 2. 初始化模型并载入最佳权重
    model = RTFEN(in_channels=1, L=32, T=50, original_size=(128, 128)).to(device)
    weights_path = 'C:/Users/rapstar/Desktop/code/ruoxinhao/saved_models/best_RTFEN_model.pth'
    model.load_state_dict(torch.load(weights_path, map_location=device))
    
    # 3. 传入局部切片边界，让预处理函数只在局部计算中位数特征对齐
    input_tensor = transform_real_pc_to_model_input(real_data, r_start=1200, r_end=1712, max_db=40.0)[:, :, :50, :, :]
    
    # 4. 执行推理
    predict_and_locate_targets_roi(model, input_tensor, device, thresh=0.5, r_start=1200, r_end=1712)

🔮 局部靶向对齐完毕！实际推理形状: torch.Size([1, 1, 50, 512, 128])，正在检测...

================== 🏆 实测数据绝对物理坐标报告 ==================
帧 [01/50] ── 🎯 成功捕获目标 | 绝对坐标: (距离单元: 1210, 脉冲: 20), (距离单元: 1238, 脉冲: 110), (距离单元: 1267, 脉冲: 9), (距离单元: 1276, 脉冲: 79)
帧 [02/50] ── 🎯 成功捕获目标 | 绝对坐标: (距离单元: 1209, 脉冲: 97), (距离单元: 1210, 脉冲: 58), (距离单元: 1248, 脉冲: 95), (距离单元: 1265, 脉冲: 84), (距离单元: 1268, 脉冲: 115), (距离单元: 1273, 脉冲: 41), (距离单元: 1274, 脉冲: 88), (距离单元: 1291, 脉冲: 20), (距离单元: 1314, 脉冲: 30)
帧 [03/50] ── 🎯 成功捕获目标 | 绝对坐标: (距离单元: 1200, 脉冲: 111), (距离单元: 1204, 脉冲: 122), (距离单元: 1213, 脉冲: 87), (距离单元: 1215, 脉冲: 32), (距离单元: 1233, 脉冲: 54), (距离单元: 1279, 脉冲: 11), (距离单元: 1285, 脉冲: 114), (距离单元: 1317, 脉冲: 118)
帧 [04/50] ── 🎯 成功捕获目标 | 绝对坐标: (距离单元: 1215, 脉冲: 87), (距离单元: 1289, 脉冲: 14), (距离单元: 1296, 脉冲: 74), (距离单元: 1313, 脉冲: 118)
帧 [05/50] ── 🎯 成功捕获目标 | 绝对坐标: (距离单元: 1201, 脉冲: 82), (距离单元: 1202, 脉冲: 36), (距离单元: 1206, 脉冲: 100), (距离单元: 1208, 脉冲: 77), (距离单元: 1209, 脉冲: 104), (距离单元: 1214, 脉冲: 62), (距离单元: 1221, 脉冲: 87), (距离单元: 1224, 脉冲: 123), (距离单元

In [11]:
a = 20 * np.log10(abs(np.fft.fftshift(np.fft.fft(real_data, axis=1), axes=1)))

# 划分有效区间： db > -50
clutter_median = np.median(a[a > -50])
zero_db_level = clutter_median - 10

a - zero_db_level

array([[[-296.70459645, -293.78492783, -298.32866283, ...,
         -294.2808575 , -299.34648332, -295.08429568],
        [-297.57045717, -303.82123803, -294.33608521, ...,
         -289.91311338, -298.97399627, -297.67791712],
        [-294.55755387, -300.5022079 , -297.56331117, ...,
         -298.44620411, -298.16141807, -297.65700746],
        ...,
        [-299.35543616, -303.43131744, -294.79121163, ...,
         -298.55602894, -299.28418844, -297.37883793],
        [-297.70448214, -296.4327709 , -308.0934886 , ...,
         -307.78003764, -289.88122036, -291.05464189],
        [-304.94433526, -295.56468607, -300.42312015, ...,
         -303.1856675 , -298.00276785, -290.2100048 ]],

       [[-295.82290916, -298.80464576, -295.5567455 , ...,
         -292.81993709, -292.70023919, -294.67864248],
        [-294.39297192, -292.03789712, -299.2974173 , ...,
         -290.89108841, -290.51134207, -290.7766275 ],
        [-290.67353085, -290.71104119, -303.88808718, ...,
         -315.

In [12]:
a

array([[[-217.67870322, -214.7590346 , -219.3027696 , ...,
         -215.25496427, -220.32059009, -216.05840245],
        [-218.54456394, -224.7953448 , -215.31019198, ...,
         -210.88722015, -219.94810304, -218.65202389],
        [-215.53166064, -221.47631467, -218.53741794, ...,
         -219.42031088, -219.13552484, -218.63111423],
        ...,
        [-220.32954293, -224.40542421, -215.7653184 , ...,
         -219.53013571, -220.25829521, -218.3529447 ],
        [-218.67858891, -217.40687767, -229.06759537, ...,
         -228.75414441, -210.85532713, -212.02874866],
        [-225.91844203, -216.53879284, -221.39722692, ...,
         -224.15977427, -218.97687462, -211.18411157]],

       [[-216.79701593, -219.77875253, -216.53085227, ...,
         -213.79404386, -213.67434596, -215.65274925],
        [-215.36707869, -213.01200389, -220.27152407, ...,
         -211.86519518, -211.48544884, -211.75073427],
        [-211.64763762, -211.68514796, -224.86219395, ...,
         -236.

In [ ]:
# === 在模型推理前，先看看数据的真实面貌 ===
    # 1. 打印实测数据的数值范围
real_max = real_data.max().item()
real_min = real_data.min().item()
real_mean = real_data.mean().item()
    
print("\n========== 🩺 数据体检报告 ==========")
print(f"实测数据 dB 范围: [{real_min:.2f}, {real_max:.2f}]")
print(f"实测数据 dB 均值: {real_mean:.2f}")
print("=====================================\n")

In [ ]:
file_path_loc = 'C:/Users/rapstar/Desktop/data/位置文件/real_target_9_20240723-082719-887_IF_CH1_400MHz_0.mat'
location_data = sio.loadmat(file_path_loc)
var_names_loc = [key for key in location_data.keys() if not key.startswith('__')]
print("位置文件中的变量：", var_names_loc)
real_target = location_data['tar_re']
print("位置数据类型：", type(real_target))  # 通常是 numpy.ndarray
print("位置数据形状：", real_target.shape)
print("位置数据内容：", real_target)

In [2]:
import numpy as np
import torch

def transform_real_pc_to_model_input(raw_complex_data, max_db=40.0):
    """
    将实测的脉压时域复数三维张量，完美驯化并对齐为可直接喂给模型的相对值频域图像（RD图）。
    
    参数:
        raw_complex_data: numpy 复数张量，形状为 [距离单元(H), 脉冲数(W), CPI数(T)]
        max_db: 训练数据对齐的绝对能量上限，默认 40.0 dB (基于论文低海况参数)
    返回:
        model_ready_tensor: torch.Tensor, 形状为 [1, 1, T, H, W]，完美符合模型 forward 的输入要求
    """
    # ------------------ 1. 物理维度对齐 ------------------
    # 原始形状: [Range(H), Pulse(W), CPI(T)]
    # 转换后形状: [CPI(T), Range(H), Pulse(W)] -> 适应模型时序序列灰度图的要求
    data = np.transpose(raw_complex_data, (2, 0, 1))
    
    # ------------------ 2. 硬件级直流偏置消除 (DC Offset Removal) ------------------
    # 针对你测出的 362+213j 硬件固定电平，在慢时间（Pulse维，即axis=2）上减去均值。
    # 这样可以绝对避免 FFT 后在零多普勒轴上留下一条毁灭性的“垂直光柱”干扰目标。
    data_centered = data - np.mean(data, axis=2, keepdims=True)
    
    # ------------------ 3. 时域变频域 (2D-FFT 变换得到 RD 图) ------------------
    # 只对慢时间（Pulse维）做 FFT，将时域上的复数震荡信号，聚焦到多普勒频率轴上
    rdm_complex = np.fft.fft(data_centered, axis=2)
    # 将零多普勒频移到中心（频谱居中），形成标准的 Range-Doppler 图像
    rdm_complex_shifted = np.fft.fftshift(rdm_complex, axes=2)
    
    # ------------------ 4. 复数变实数矩阵 (CV 灰度图能量化) ------------------
    # 提取绝对幅度，此时数据彻底由 Complex 转化为可用于图像处理的 Real 数值
    rdm_amp = np.abs(rdm_complex_shifted)
    
    # 转换为雷达视觉任务标配的对数分贝（dB）图，这样能够极大程度保留并拉高暗弱海杂波的时空纹理
    rdm_db = 20 * np.log10(rdm_amp + 1e-8)
    
    # ------------------ 5. 绝对值变相对值 (对齐训练数据的物理底色) ------------------
    # 实测数据的绝对 dB（比如 70dB 以上）模型不认识，而模拟数据海杂波均值在 10dB 左右
    # 我们用中位数（Median）稳健地抓出当前实测海面的平均海浪底色
    clutter_median = np.median(rdm_db)
    
    # 反推实测环境下的 0 dB 纯噪声电平：
    zero_db_level = clutter_median - 10.0
    
    # 实施平移，将画面整体拉回至以模拟底噪为基准的相对刻度轴上
    rdm_db_relative = rdm_db - zero_db_level
    
    # ------------------ 6. 动态范围裁剪与最终 CV 归一化 ------------------
    # 严格按照训练时的逻辑：x = torch.clamp(x, 0.0, 40.0) / 40.0
    # 这样微弱噪声和盲区会被 clip 成 0.0（纯黑），海杂波背景处于 0.25 左右（有细节的灰度层），
    # 真正被多普勒相干积累捞出来的强目标会逼近并截断在 1.0（最高高光点）。
    rdm_normalized = np.clip(rdm_db_relative, 0.0, max_db) / max_db
    
    # ------------------ 7. 补充深度学习所需的 Batch 和 Channel 维度 ------------------
    # 从 [T, H, W] 补齐至 [Batch=1, Channel=1, T, H, W]
    model_ready_tensor = torch.from_numpy(rdm_normalized).float().unsqueeze(0).unsqueeze(0)
    
    return model_ready_tensor

In [ ]:
import os
# ✅ Windows平台专用内存优化配置（完全支持，无警告）
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256,garbage_collection_threshold:0.8"
# 解决OMP冲突
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import numpy as np
import torch
from scipy.ndimage import label, center_of_mass
from functions import RTFEN

def transform_real_pc_to_model_input(raw_complex_data, max_db=40.0):
    """
    将实测的脉压时域复数三维张量，完美驯化并对齐为可直接喂给模型的相对值频域图像（RD图）。
    
    参数:
        raw_complex_data: numpy 复数张量，形状为 [距离单元(H), 脉冲数(W), CPI数(T)]
        max_db: 训练数据对齐的绝对能量上限，默认 40.0 dB (基于论文低海况参数)
    返回:
        model_ready_tensor: torch.Tensor, 形状为 [1, 1, T, H, W]，完美符合模型 forward 的输入要求
    """
    # ------------------ 1. 物理维度对齐 ------------------
    # 原始形状: [Range(H), Pulse(W), CPI(T)]
    # 转换后形状: [CPI(T), Range(H), Pulse(W)] -> 适应模型时序序列灰度图的要求
    data = np.transpose(raw_complex_data, (2, 0, 1))
    
    # ------------------ 2. 硬件级直流偏置消除 (DC Offset Removal) ------------------
    # 针对你测出的 362+213j 硬件固定电平，在慢时间（Pulse维，即axis=2）上减去均值。
    # 这样可以绝对避免 FFT 后在零多普勒轴上留下一条毁灭性的“垂直光柱”干扰目标。
    data_centered = data - np.mean(data, axis=2, keepdims=True)
    
    # ------------------ 3. 时域变频域 (2D-FFT 变换得到 RD 图) ------------------
    # 只对慢时间（Pulse维）做 FFT，将时域上的复数震荡信号，聚焦到多普勒频率轴上
    rdm_complex = np.fft.fft(data_centered, axis=2)
    # 将零多普勒频移到中心（频谱居中），形成标准的 Range-Doppler 图像
    rdm_complex_shifted = np.fft.fftshift(rdm_complex, axes=2)
    
    # ------------------ 4. 复数变实数矩阵 (CV 灰度图能量化) ------------------
    # 提取绝对幅度，此时数据彻底由 Complex 转化为可用于图像处理的 Real 数值
    rdm_amp = np.abs(rdm_complex_shifted)
    
    # 转换为雷达视觉任务标配的对数分贝（dB）图，这样能够极大程度保留并拉高暗弱海杂波的时空纹理
    rdm_db = 20 * np.log10(rdm_amp + 1e-8)
    
    # ------------------ 5. 绝对值变相对值 (对齐训练数据的物理底色) ------------------
    # 实测数据的绝对 dB（比如 70dB 以上）模型不认识，而模拟数据海杂波均值在 10dB 左右
    # 我们用中位数（Median）稳健地抓出当前实测海面的平均海浪底色
    clutter_median = np.median(rdm_db)
    
    # 反推实测环境下的 0 dB 纯噪声电平：
    zero_db_level = clutter_median - 10.0
    
    # 实施平移，将画面整体拉回至以模拟底噪为基准的相对刻度轴上
    rdm_db_relative = rdm_db - zero_db_level
    
    # ------------------ 6. 动态范围裁剪与最终 CV 归一化 ------------------
    # 严格按照训练时的逻辑：x = torch.clamp(x, 0.0, 40.0) / 40.0
    # 这样微弱噪声和盲区会被 clip 成 0.0（纯黑），海杂波背景处于 0.25 左右（有细节的灰度层），
    # 真正被多普勒相干积累捞出来的强目标会逼近并截断在 1.0（最高高光点）。
    rdm_normalized = np.clip(rdm_db_relative, 0.0, max_db) / max_db
    
    # ------------------ 7. 补充深度学习所需的 Batch 和 Channel 维度 ------------------
    # 从 [T, H, W] 补齐至 [Batch=1, Channel=1, T, H, W]
    model_ready_tensor = torch.from_numpy(rdm_normalized).float().unsqueeze(0).unsqueeze(0)
    
    return model_ready_tensor



def predict_and_locate_targets_roi(model, model_input, device, thresh=0.5, r_start=1200, r_end=1712):
    """
    让模型对空间裁剪后的 ROI 数据进行推理，并自动还原、打印出目标在 2662 大图里的绝对物理坐标。
    
    参数:
        model: RTFEN 模型实例
        model_input: 形状为 [1, 1, 80, 2662, 128] 的全量实测数据张量
        device: 'cuda' 或 'cpu'
        thresh: 二值化阈值 (0.5)
        r_start, r_end: 切片裁剪的边界 (1400:1656，长度为 256)
    """
    model.eval()
    
    # 🌟 核心修改1：在 CPU/内存中先完成轻量化切片，只把 256 大小的紧凑小张量送入显存！
    # 彻底解放显存，12G 显卡可以极其丝滑地秒跑完
    roi_input = model_input[:, :, :, r_start:r_end, :].to(device)
    
    print(f"🔮 RTFEN 显存危机已解除！实际推理形状: {roi_input.shape}，正在深度检测...")
    
    with torch.no_grad():
        # outputs 形状: [1, 80, 1, 256, 128]
        outputs = model(roi_input)
        
        # 1. 阈值二值化处理
        pred_mask = (outputs > thresh).float()
        
        # 2. 剥离 Batch 和 Channel 维度，转回 CPU NumPy 做连通域质心提取
        # pred_mask_np 形状: [80, 256, 128] -> [T, ROI_Range, Pulse]
        pred_mask_np = pred_mask.squeeze().cpu().numpy()

    print("\n================== 🏆 实测数据绝对物理坐标报告 ==================")
    
    total_frames = pred_mask_np.shape[0] # 80 帧
    
    for t in range(total_frames):
        frame_mask = pred_mask_np[t, :, :] # 提取当前帧的 2D 画面 [256, 128]
        
        # 3. 使用 8-连通域分析（CCA）对当前 256×128 的小画面进行“岛屿”计数
        labeled_mask, num_features = label(frame_mask, structure=np.ones((3, 3)))
        
        if num_features == 0:
            print(f"帧 [{t+1:02d}/80] ── ⚠️ 未检测到有效目标。")
            continue
            
        # 4. 计算当前帧中所有预测目标的连通域质心（相对 256 窗的局部坐标）
        centroids = center_of_mass(frame_mask, labeled_mask, range(1, num_features + 1))
        
        detected_targets = []
        for c in centroids:
            if not np.isnan(c[0]):
                # c[0] 是在 256 窗内部的相对行号（距离），c[1] 是相对列号（脉冲）
                local_range_idx = int(round(c[0]))
                pulse_idx = int(round(c[1]))
                
                # 🌟 核心修改2：通过加上 r_start(1200)，将局部相对坐标无损还原为原始 2662 大图的绝对物理坐标！
                absolute_range_idx = local_range_idx + r_start
                
                detected_targets.append((absolute_range_idx, pulse_idx))
        
        # 5. 格式化打印当前帧的真实绝对坐标
        if len(detected_targets) > 0:
            target_str = ", ".join([f"(距离单元: {r}, 脉冲: {p})" for r, p in detected_targets])
            print(f"帧 [{t+1:02d}/80] ── 🎯 成功捕获目标数量: {len(detected_targets)} | 绝对坐标: {target_str}")
        else:
            print(f"帧 [{t+1:02d}/80] ── ⚠️ 未检测到有效目标。")

    print("=================================================================\n")

# ================= 调用推理 =================
# 加载模型权重
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RTFEN(in_channels=1, L=32, T=50, original_size=(128, 128)).to(device) # 注意：这里如果为了适应大图，确保RTFEN模型能跑下非128x128的输入
model.load_state_dict(torch.load('C:/Users/rapstar/Desktop/code/ruoxinhao/saved_models/best_RTFEN_model.pth', map_location=device))

# 执行检测定位
input_data = transform_real_pc_to_model_input(real_data)[:, :, :50, :, :]
predict_and_locate_targets_roi(model, input_data, device, thresh=0.8, r_start=1200, r_end=1712)

🔮 RTFEN 显存危机已解除！实际推理形状: torch.Size([1, 1, 50, 512, 128])，正在深度检测...

================== 🏆 实测数据绝对物理坐标报告 ==================
帧 [01/80] ── 🎯 成功捕获目标数量: 40 | 绝对坐标: (距离单元: 1201, 脉冲: 44), (距离单元: 1204, 脉冲: 27), (距离单元: 1208, 脉冲: 110), (距离单元: 1224, 脉冲: 115), (距离单元: 1225, 脉冲: 84), (距离单元: 1226, 脉冲: 26), (距离单元: 1227, 脉冲: 51), (距离单元: 1232, 脉冲: 111), (距离单元: 1234, 脉冲: 75), (距离单元: 1235, 脉冲: 18), (距离单元: 1237, 脉冲: 96), (距离单元: 1238, 脉冲: 62), (距离单元: 1238, 脉冲: 101), (距离单元: 1240, 脉冲: 38), (距离单元: 1242, 脉冲: 88), (距离单元: 1246, 脉冲: 43), (距离单元: 1253, 脉冲: 106), (距离单元: 1256, 脉冲: 48), (距离单元: 1260, 脉冲: 94), (距离单元: 1262, 脉冲: 122), (距离单元: 1263, 脉冲: 31), (距离单元: 1265, 脉冲: 2), (距离单元: 1265, 脉冲: 55), (距离单元: 1269, 脉冲: 21), (距离单元: 1279, 脉冲: 40), (距离单元: 1284, 脉冲: 94), (距离单元: 1285, 脉冲: 57), (距离单元: 1285, 脉冲: 100), (距离单元: 1297, 脉冲: 25), (距离单元: 1299, 脉冲: 12), (距离单元: 1301, 脉冲: 38), (距离单元: 1305, 脉冲: 52), (距离单元: 1313, 脉冲: 40), (距离单元: 1313, 脉冲: 51), (距离单元: 1315, 脉冲: 27), (距离单元: 1318, 脉冲: 68), (距离单元: 1320, 脉冲: 54), (距离单元: 1322, 脉冲: 29), (

In [ ]:
a = np.fft.fft(real_data[:,:,0])
np.clip(20 * np.log10(np.abs(a) + 1e-8), 0.0, 40.0) / 40.0

In [ ]:
import numpy as np
import torch

def process_real_data_to_cv_image(raw_data_complex):
    """
    严格按照 CV 语义分割任务的要求，将实测时域复数转化为相对值 RD 图。
    """
    # 1. 维度对齐：[Range, Pulse, CPI] -> [CPI(T), Range(H), Pulse(W)]
    # 让它符合模型序列图像的要求
    data = np.transpose(raw_data_complex, (2, 0, 1))
    
    # 【非常关键的预处理】：去直流偏置。实测的复数域均值(比如你之前的 362+213j)
    # 如果不减掉，FFT 之后图像正中间会有一条极亮的垂直线，毁掉整个画面。
    data = data - np.mean(data, axis=2, keepdims=True)
    
    # 2. 时域 -> 频域 (FFT 变换得到 Range-Doppler 图)
    # 这就是你说的：把时域变换到频域来
    rdm_complex = np.fft.fftshift(np.fft.fft(data, axis=2), axes=2)
    
    # 3. 复数 -> 实数矩阵
    # 把复数取绝对值，也就是图像上每个像素的“灰度值/能量值”
    rdm_amp = np.abs(rdm_complex)
    
    # 将幅度转为对数 dB（雷达 CV 任务中最常用的做法，能凸显微弱的杂波细节）
    rdm_db = 20 * np.log10(rdm_amp + 1e-8)
    
    # 4. 绝对值 -> 相对值 (核心评估对齐步)
    # 实测数据的 dB 值绝对刻度（比如 60dB~120dB）模型不认识。
    # 我们找整个画面的中位数作为背景底色（海浪），把它强行拉回 0 的基准线
    noise_floor = np.median(rdm_db)
    rdm_db_relative = rdm_db - noise_floor 
    
    # 最后一步：把它压缩成模拟数据在训练时的那种相对对比度 (比如 40dB 的动态范围截断)
    # 这样背景海浪的值大约在 0.2 左右，强目标在 1.0 左右。
    # 此时，实测数据就彻底变成了一张标准化的“CV 测试集图像”！
    rdm_normalized = np.clip(rdm_db_relative, 0.0, 40.0) / 40.0
    
    # 补充模型需要的 Batch 和 Channel 维度 -> [1, 1, T, H, W]
    model_input = torch.from_numpy(rdm_normalized).float().unsqueeze(0).unsqueeze(0)
    
    return rdm_complex, rdm_amp, model_input

complex, amp, res = process_real_data_to_cv_image(real_data)

In [ ]:
import h5py
file = h5py.File('C:/Users/rapstar/Desktop/code/ruoxinhao/radar_dataset_30_50.h5', "r")
test_data = file['rdm_data'][:]
mask = file['mask_data'][:]
file.close()

In [ ]:
x = np.clip(test_data, 0.0, 40.0) / 40.0
x

In [9]:
import os
import h5py
import numpy as np
import torch
import math
from scipy.ndimage import label, center_of_mass
from torch.utils.data import Dataset, DataLoader

# ====================== 你的模型文件必须能导入 ======================
from functions import RTFEN, TemporalTverskyLoss

# ====================== 测试配置（只改这里！）======================
TEST_DATA_PATH = 'C:/Users/rapstar/Desktop/code/ruoxinhao/radar_dataset_30_50.h5'  # 新测试集路径
MODEL_PATH      = 'C:/Users/rapstar/Desktop/code/ruoxinhao/saved_models/best_RTFEN_model.pth'
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE      = 4
THRESH          = 0.5

# ====================== 数据加载 ======================
class RadarH5Dataset(Dataset):
    def __init__(self, rdm_tensor, mask_tensor):
        self.rdm_tensor = rdm_tensor
        self.mask_tensor = mask_tensor

    def __len__(self):
        return self.rdm_tensor.shape[0]

    def __getitem__(self, idx):
        x = self.rdm_tensor[idx]
        y = self.mask_tensor[idx]

        x = torch.clamp(x, 0.0, 40.0) / 40.0
        x_tensor = x.float().unsqueeze(0)
        y_tensor = y.float().unsqueeze(1)

        return x_tensor, y_tensor

# ====================== CCA 质心评估（完全和你训练一致）======================
def evaluate_frame_with_cca(pred_mask_numpy, true_mask_numpy, distance_threshold=1.5):
    pred_labeled, pred_num_features = label(pred_mask_numpy, structure=np.ones((3,3)))
    true_labeled, true_num_features = label(true_mask_numpy, structure=np.ones((3,3)))
    
    pred_centroids = center_of_mass(pred_mask_numpy, pred_labeled, range(1, pred_num_features + 1))
    true_centroids = center_of_mass(true_mask_numpy, true_labeled, range(1, true_num_features + 1))
    
    pred_centroids = [c for c in pred_centroids if not np.isnan(c[0])]
    true_centroids = [c for c in true_centroids if not np.isnan(c[0])]

    TP, FP, FN = 0, 0, 0
    if len(true_centroids) == 0:
        return 0, len(pred_centroids), 0
    if len(pred_centroids) == 0:
        return 0, 0, len(true_centroids)

    matched_true_indices = set()
    for pred_c in pred_centroids:
        min_dist = float('inf')
        best_true_idx = -1
        for i, true_c in enumerate(true_centroids):
            dist = math.sqrt((pred_c[0] - true_c[0])**2 + (pred_c[1] - true_c[1])**2)
            if dist < min_dist:
                min_dist = dist
                best_true_idx = i
        if min_dist <= distance_threshold and best_true_idx not in matched_true_indices:
            TP += 1
            matched_true_indices.add(best_true_idx)
        else:
            FP += 1
    FN = len(true_centroids) - len(matched_true_indices)
    return TP, FP, FN

# ====================== 模型评估 ======================
def evaluate(model, dataloader, device, thresh=0.5):
    model.eval()
    total_tp, total_fp, total_fn = 0, 0, 0

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            preds_bin = (outputs > thresh).float().cpu().numpy()
            true_masks = targets.cpu().numpy()

            for b in range(preds_bin.shape[0]):
                for t in range(preds_bin.shape[1]):
                    tp, fp, fn = evaluate_frame_with_cca(
                        preds_bin[b, t, 0], true_masks[b, t, 0], distance_threshold=1.5
                    )
                    total_tp += tp
                    total_fp += fp
                    total_fn += fn

    precision = total_tp / (total_tp + total_fp + 1e-8)
    recall = total_tp / (total_tp + total_fn + 1e-8)
    f1_score = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1_score

# ====================== 主测试流程 ======================
def main():
    print("🔹 加载测试数据...")
    with h5py.File(TEST_DATA_PATH, 'r') as f:
        rdm_test  = torch.from_numpy(f['rdm_data'][:]).float()
        mask_test = torch.from_numpy(f['mask_data'][:]).float()

    test_dataset = RadarH5Dataset(rdm_test, mask_test)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

    print("🔹 加载模型...")
    model = RTFEN(in_channels=1, L=32, T=50, original_size=(128, 128)).to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()

    print("🔹 开始测试...")
    p, r, f1 = evaluate(model, test_loader, DEVICE, THRESH)

    print("\n" + "="*50)
    print("🎉 测试完成 🎉")
    print(f"Precision: {p*100:.2f}%")
    print(f"Recall:    {r*100:.2f}%")
    print(f"F1-Score:  {f1*100:.2f}%")
    print("="*50)

if __name__ == '__main__':
    main()

🔹 加载测试数据...
🔹 加载模型...
🔹 开始测试...


KeyboardInterrupt: 